In [156]:
import requests
import os
# from langchain_openai import ChatOpenAI
from typing import TypedDict
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
# from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI
# from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
# from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages
# from pydantic import BaseModel, Field
from langchain_core.pydantic_v1 import BaseModel, Field
from IPython.display import Image, display
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
import gradio as gr
import uuid
from dotenv import load_dotenv

In [157]:
from langgraph.checkpoint.sqlite import SqliteSaver

# memory = SqliteSaver.from_conn_string(":memory:")
memory = MemorySaver()

In [158]:
PROJECT_ID = os.getenv("VERTEX_PROJECT_ID")
REGION = os.getenv("VERTEX_REGION")

In [ ]:
# RAG API configuration
RAG_API_URL = os.getenv("RAG_API_URL", "http://localhost:9000")
RAG_API_TIMEOUT = int(os.getenv("RAG_API_TIMEOUT", "10"))


In [159]:
load_dotenv(override=True)

True

In [160]:
llm = ChatVertexAI(
    model="gemini-2.5-flash",
    project=PROJECT_ID,
    location=REGION
)

In [161]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        
        # Add tool handling if tools are provided
        if tools:
            graph.add_node("tools", self.take_action)
            graph.add_conditional_edges("llm", self.should_continue)
            graph.add_edge("tools", "llm")
        else:
            graph.add_edge("llm", END)
        
        graph.add_edge(START, "llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools} if tools else {}
        # Bind tools to model if tools are provided
        self.model = model.bind_tools(tools) if tools else model

    def call_llm(self, state: AgentState):
        messages = state['messages']
        if self.system:
            # Only add system message if not already present in messages
            if not messages or not isinstance(messages[0], SystemMessage):
                messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def should_continue(self, state: AgentState):
        """Determine if we should continue to tools or end."""
        messages = state['messages']
        last_message = messages[-1]
        # Check if the last message has tool calls
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            return "tools"
        return END

    def take_action(self, state: AgentState):
        """Execute tool calls from the last message."""
        messages = state['messages']
        last_message = messages[-1]
        tool_calls = last_message.tool_calls
        results = []
        for tool_call in tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call.get('args', {})
            tool_call_id = tool_call.get('id', str(uuid.uuid4()))
            
            if tool_name in self.tools:
                try:
                    print(f"[TOOL] Calling {tool_name} with args: {tool_args}")
                    result = self.tools[tool_name].invoke(tool_args)
                    results.append(ToolMessage(
                        tool_call_id=tool_call_id,
                        name=tool_name,
                        content=str(result)
                    ))
                except Exception as e:
                    print(f"[ERROR] Tool {tool_name} failed: {e}")
                    results.append(ToolMessage(
                        tool_call_id=tool_call_id,
                        name=tool_name,
                        content=f"Error: {str(e)}"
                    ))
            else:
                results.append(ToolMessage(
                    tool_call_id=tool_call_id,
                    name=tool_name,
                    content=f"Tool {tool_name} not found"
                ))
        return {'messages': results}

In [163]:
class preference(BaseModel):
    """User investment preferences"""
    long_term: bool = Field(description="Long term investment preference.")
    short_term: bool = Field(description="Short term investment preference.")
    high_risk: bool = Field(description="High risk appetite check.")
    low_risk: bool = Field(description="Low risk appetite check.")
    sectors: list = Field(description="Preferred investment sectors.")

    # name: str = Field(description="The full name of the recipe.")
    # servings: int = Field(description="The number of people the recipe serves.")
    # prep_time_minutes: Optional[int] = Field(None, description="The estimated preparation time in minutes.")
    # ingredients: List[Ingredient] = Field(description="A list of all required ingredients.")

In [ ]:
# RAG Query Tool for financial knowledge base
def query_financial_knowledge_base(query: str) -> str:
    """Query the financial knowledge base for definitions, explanations, and financial concepts.
    
    Use this tool when the user asks about:
    - Financial terms (e.g., P/E ratio, quantitative momentum, market cap)
    - Investment concepts
    - Financial metrics and their meanings
    - Feature definitions and thresholds from the quantitative model
    
    Args:
        query: The financial question or term to look up
        
    Returns:
        A string containing relevant information from the knowledge base
    """
    try:
        response = requests.post(
            f"{RAG_API_URL}/query/text",
            json={"q": query, "k": 3, "format": "text"},
            timeout=RAG_API_TIMEOUT
        )
        if response.status_code == 200:
            data = response.json()
            if data.get("found"):
                return data.get("answer", "No information found.")
            else:
                return "No relevant information found in knowledge base."
        else:
            return f"Knowledge base unavailable (status: {response.status_code})"
    except requests.Timeout:
        return "Knowledge base query timed out. Please try again."
    except requests.ConnectionError:
        return "Could not connect to knowledge base. Please ensure the RAG service is running."
    except Exception as e:
        return f"Error accessing knowledge base: {str(e)}"

# Create LangChain tool from the function
from langchain_core.tools import tool

rag_tool = tool(query_financial_knowledge_base)


In [146]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
system_prompt = """You are an AI assistant which is collecting financial requirements from a user. Start the conversation by introducing yourself. Be polite and ask about their name in a welcoming tone.

You have access to a financial knowledge base tool that can provide definitions and explanations of financial terms, investment concepts, and quantitative model features. Use this tool when users ask questions about financial concepts or when you need to provide accurate explanations.

You will be asking questions one by one and if you don't get the answer or are asked a clarifying query for your input, answer by explaining politely.

Here are the questions:

Q1: What is your risk appetite? low, medium, high?
Q2: What is your investment horizon? short term (less than 3 months), more than 3 months.
Q3: Is there any particular sector you are interested in?

When users ask about financial terms or concepts you're unsure about, use the query_financial_knowledge_base tool to get accurate information.

When you are able to answer all the questions, you may end conversation by showing the final output.

**CRITICAL INSTRUCTION: The final output must be a single summary containing all collected data in the following exact format:**
### Final Financial Requirements
{
    long_term: bool = Field(description="Long term investment preference.")
    short_term: bool = Field(description="Short term investment preference.")
    high_risk: bool = Field(description="High risk appetite check.")
    low_risk: bool = Field(description="Low risk appetite check.")
    sectors: list = Field(description="Preferred investment sectors.")
}
"""

In [148]:
# # model = ChatOpenAI(model="gpt-4o")
# llm_structured = llm.with_structured_output(preference)
# abot = Agent(llm_structured, [], system=system_prompt, checkpointer=memory)

In [ ]:
# model = ChatOpenAI(model="gpt-4o")
# llm_structured = llm.with_structured_output(preference)
# abot = Agent(llm_structured, [], system=system_prompt, checkpointer=memory)

In [ ]:
# model = ChatOpenAI(model="gpt-4o")
# llm_structured = llm.with_structured_output(preference)
# Create agent with RAG tool for financial knowledge base queries
abot = Agent(llm, [rag_tool], system=system_prompt, checkpointer=memory)

In [182]:
config = {"configurable": {"thread_id": "1"}}
async def chat(user_input: str, history):
    print(history)
    result = await abot.graph.ainvoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content

In [183]:
config = {"configurable": {"thread_id": "1"}}
history = []
def chat(user_input: str, history):
    
    # if not history:
    #     return "Hello! How can I assist you today?" 
    result = abot.graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    print(result)
    return result["messages"][-1].content

In [ ]:
res1 = gr.ChatInterface(fn=chat,
                type="messages",
                # chatbot=gr.Chatbot(placeholder="<strong>Your Personal Yes-Man</strong><br>Ask Me Anything"),
                description="Ask AI chat Assistant"
                 ).launch()

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.


{'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='ba4d9643-8b85-47af-a247-9a83da2f1cb7'), AIMessage(content="Hello! I'm an AI assistant designed to help you gather your financial requirements. I'll ask you a few questions to understand your investment preferences.\n\nTo start, could you please tell me your name?", additional_kwargs={}, response_metadata={'is_blocked': False, 'safety_ratings': [], 'usage_metadata': {'prompt_token_count': 267, 'candidates_token_count': 43, 'total_token_count': 334, 'prompt_tokens_details': [{'modality': 1, 'token_count': 267}], 'candidates_tokens_details': [{'modality': 1, 'token_count': 43}], 'thoughts_token_count': 24, 'cached_content_token_count': 0, 'cache_tokens_details': []}, 'finish_reason': 'STOP', 'avg_logprobs': -0.29126224961391717, 'model_name': 'gemini-2.5-flash'}, id='run--d2579e9c-f8ce-484d-9d5c-4d109c4ada42-0', usage_metadata={'input_tokens': 267, 'output_tokens': 43, 'total_tokens': 334, 'input_toke

In [169]:
result

NameError: name 'result' is not defined